In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc 
import anndata as ad
#import besca as bc
import h5py 
import glob
import matplotlib.pyplot as plt
import os

In [ ]:
sc.settings.verbosity = 3             
sc.logging.print_header()
sc.settings.set_figure_params(dpi=80, facecolor='white')
sc.set_figure_params(scanpy=True, figsize=(8,8))      

In [ ]:
# Set base path
base_path = '/home/EOCRC_atlas/'

In [ ]:
# load the atlas
adata = sc.read_h5ad(os.path.join(base_path, 'data/all_samples_processed.h5ad'))

In [ ]:
print(adata.layers['counts'][0:20,0:20]) # check that the counts layer are integers 
print(adata.raw.X[0:20,0:20]) # check that normalized values are as expected

In [ ]:
# Tier 1 annotations using Kong et al markers 
# Define Lingjia's signatures, calculate scores, save adata, plot on UMAP 
kong_markers = {
    'epithelial' : ['EPCAM', 'KRT8', 'KRT18'],
    'stromal' : ['CDH5', 'COL1A1', 'COL1A2', 'COL6A2', 'VWF'],
    'immune' : ['PTPRC', 'CD3D', 'CD3G', 'CD3E', 'CD79A', 'CD79B', 'CD14', 'CD16', 'CD68', 'CD83', 'CSF1R', 'FCER1G'],
    'enterocytes' : ['RBP2', 'ANPEP', 'FABP2'],
    'stem_cells' : ['LGR5', 'ASCL2', 'SMOC2', 'RGMB', 'OLFM4'],
    'goblets' : ['CLCA1', 'SPDEF', 'FCGBP', 'ZG16', 'MUC2'],
    'paneth' : ['DEFA5', 'DEFA6', 'REG3A'],
    'tuft' : ['LRMP', 'SH2D6'],
    'enteroendocrine' : ['CHGA', 'CHGB', 'NEUROD1'],
    'cycling' : ['UBE2C', 'TOP2A', 'MKI67', 'HMGB2'],
    'fibroblasts' : ['ADAMDEC1', 'PDGFRA', 'BMP4'],
    'myofibroblasts' : ['TAGLN', 'ACTG2'],
    'lymphatics' : ['CCL21', 'TFF3'],
    'endothelial' : ['CD36', 'ACKR174'],
    'pericytes' : ['NOTCH3', 'CD146', 'RGS5'],
    'glial' : ['FOXD3', 'MPZ', 'CDH19', 'PLP1', 'SOX10', 'S100B', 'ERBB3'],
    't' : ['CD3D', 'CD3G', 'CD3E'],
    'b' : ['CD79A', 'CD20', 'CD19'],
    'myeloid_cells' : ['CD14', 'CD16', 'HLA-DR'],
    'CD8_T' : ['CD8A', 'CD8B'],
    'CD4_T' : ['CD4'],
    'ILC' : ['RORC', 'IL1R1', 'IL23R', 'KIT', 'TNFSF4', 'PCDH9'],
    'NK' : ['EOMES', 'PRF1', 'NKG7'],
    'plasma' : ['SDC1', 'MZB1', 'SSR4', 'XBP1'],
    'b_sub' : ['BANK', 'CD30', 'ADAM28', 'VPREB3'],
    'germinal_center' : ['LRMP', 'GPT2', 'PAG1'],
    'mast' : ['GATA2', 'CPA3', 'HPGDS'],
    'macrophages' : ['CD163', 'C1QB', 'C1QC'],
    'monocytes' : ['FCN1', 'S100A4', 'S100A6'],
    'DC1' : ['CLEC9A', 'XCR1'], 
    'DC2' : ['CLEC10A', 'FCER1A'] 
}

for name, markers in kong_markers.items():
    sc.tl.score_genes(adata, markers, score_name = f'kong_{name}_markers', use_raw=True)

marker_subsets = [f'kong_{name}_markers' for name in kong_markers.keys()]
sc.pl.umap(adata, color=marker_subsets, size=2, cmap='inferno')

In [ ]:
# Look for plasma cells 
sc.pl.umap(adata, color='JCHAIN', wspace=0.1, size=2, cmap='inferno')
sc.pl.umap(adata, color='EPCAM', wspace=0.1, size=2, cmap='inferno')

In [ ]:
# Plot Leiden clusters 
sc.pl.umap(adata, color='leiden_res.1', size=1, show=True)

In [ ]:
# assign annotation to each cluster 
tier1annotation = {
    '0' : 'Epithelial', #
    '1' : 'Epithelial', #
    '2' : 'Epithelial', # HAS GLIAL COMPONENT 
    '3' : 'Epithelial', #
    '4' : 'Stromal', #
    '5' : 'Epithelial', #
    '6' : 'Myeloid', #
    '7' : 'Endothelial', #
    '8' : 'T', #
    '9' : 'Epithelial', #
    '10' : 'Epithelial', #
    '11' : 'Epithelial', #
    '12' : 'B', #
    '13' : 'Epithelial', #
    '14' : 'Epithelial', #
    '15' : 'Epithelial', #
    '16' : 'Epithelial', #
    '17' : 'Epithelial', #
    '18' : 'Epithelial', #
    '19' : 'Epithelial', #
    '20' : 'Epithelial', #
    '21' : 'Epithelial', #
    '22' : 'Epithelial', #
    '23' : 'Epithelial', #
    '24' : 'Epithelial', #
    '25' : 'Epithelial', #
    '26' : 'Epithelial', #
    '27' : 'Epithelial', #
    '28' : 'Epithelial', #
    '29' : 'Epithelial', #
    '30' : 'Glial/Neuronal'
}
adata.obs['Annotation_Tier1'] = adata.obs['leiden_res.1'].map(tier1annotation).astype('category')

In [ ]:
# optional jump in 
adata = sc.read_h5ad(os.path.join(base_path, 'data/all_samples_processed_annotated.h5ad'))

In [ ]:
# Plot lineage level annotations (Tier1) 
colors = ['#000000', '#E69F00', '#0072B2', '#D55E00', '#009E73', '#56B4E9', '#CC79A7','#F0E442',]

fig, ax = plt.subplots()
sc.pl.umap(adata, color='Annotation_Tier1', size=1, ax=ax, show=True, palette=colors)
fig.savefig(os.path.join(base_path, 'results/2025-09-26_YOCRC_annotations/Annotation_Tier1_UMAP.pdf'), dpi=600, bbox_inches='tight')
plt.close(fig)

In [ ]:
# save atlas 
adata.write_h5ad(os.path.join(base_path, 'data/all_samples_processed_annotated.h5ad'))

In [ ]:
# load raw version of the atlas 
adata_raw = sc.read_h5ad(os.path.join(base_path, 'data/all_samples_raw_withProcessedInfo.h5ad'))

In [ ]:
# check that sizes check out 
print(sum(adata_raw.obs_names==adata.obs_names))
print(adata_raw.shape)
print(adata.shape)

In [ ]:
# transfer obsm and obs to the raw data object
adata_raw.obs = adata.obs 
adata_raw.obsm = adata.obsm

In [ ]:
# save raw object with metadata & annotations
adata_raw.write_h5ad(os.path.join(base_path, 'data/all_samples_raw_withAnnotation.h5ad'))